# Krishi Vaidya: Automated Data Pipeline & Training (Colab)

This notebook runs the Krishi Vaidya 11-class YOLOv8 "Bouncer" model pipeline end-to-end on Google Colab.

## Prerequisites:
1. Add your API keys to Colab Secrets (the key icon on the left sidebar):
   - `KAGGLE_USERNAME`
   - `KAGGLE_KEY`
   - `ROBOFLOW_API_KEY`
2. Set your runtime to **T4 GPU** or better (`Runtime` -> `Change runtime type`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Setup Workspace
Navigate to the directory in your Google Drive where the `yoloml` code is stored.
*(Edit the path below to match where you uploaded your code)*

In [ ]:
import os

# TODO: Change this to the actual path in your Drive where 'yoloml' is located
PROJECT_DIR = '/content/drive/MyDrive/krishivaidya/yoloml'

# Create the directory if it doesn't exist
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

## 2. Install Dependencies
Install the core pipeline requirements and the `ultralytics` package for training.

In [ ]:
!pip install -r requirements.txt
!pip install ultralytics

## 3. Load API Credentials
Automatically load Kaggle and Roboflow credentials from Colab Secrets so they are available to the pipeline scripts.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
    print("\u2705 APIs loaded successfully from Colab Secrets.")
except Exception as e:
    print(f"\u274c Error loading secrets: {e}")
    print("Please ensure you have configured KAGGLE_USERNAME, KAGGLE_KEY, and ROBOFLOW_API_KEY in the Colab Secrets tab.")

## 4. Materialize Dataset (Auto-download & Merge)
Run the orchestrator. It will download everything from Kaggle and Roboflow via APIs, including the newly enabled `rice_leaf_kaggle`, `tomato_fruit_roboflow`, and `brassica_leaf_cauliflower_roboflow` sources, normalize annotations into the canonical Hugging Face dataset in `hf_dataset`, and also export the derived YOLO training dataset in `krishi_bouncer_dataset`.

In [ ]:
!python scripts/materialize_bouncer.py --config configs/sources.yaml --force --output-format both

## 5. Validate the Dataset
Validate both outputs before training or uploading: the canonical Hugging Face dataset and the derived YOLO dataset.

In [ ]:
!python scripts/validate.py --dataset hf_dataset --format canonical
!python scripts/validate.py --dataset krishi_bouncer_dataset --format yolo

## 6. Upload the Canonical Dataset to Hugging Face
This uploads the canonical dataset in `hf_dataset` to a Hugging Face dataset repository. Add `HF_TOKEN` to Colab Secrets first, then set `HF_REPO_ID` below.

In [ ]:
import os
from google.colab import userdata

HF_REPO_ID = 'your-username/krishi-bouncer-dataset'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

!python scripts/package_hf_dataset.py --input hf_dataset --repo-id {HF_REPO_ID}

## 7. Train the 11-Class YOLOv8 Bouncer
Launch the training using the GPU.
Adjust `epochs` based on your timing needs. `yolov8n.pt` is the nano model, designed perfectly for fast, on-device mobile inference.

In [ ]:
from ultralytics import YOLO

# Initialize YOLO model with nano architecture
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='krishi_bouncer_dataset/data.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=16, 
    name='krishi_bouncer_v1', 
    device=0, # Auto-select GPU
    patience=10 # Early stopping to prevent overfitting
)

## 8. Export for Mobile (TFLite INT8)
Once the training is done, export the `best.pt` weights to a quantized TFLite model so we can drop it straight into the React Native app's file system.

In [ ]:
!yolo export model=runs/detect/krishi_bouncer_v1/weights/best.pt format=tflite int8=True data=krishi_bouncer_dataset/data.yaml imgsz=640